# Colab GPU smoke test - Urdu OCR project

Run this **first in every Colab session**, before any training.

Before running: `Runtime > Change runtime type > Hardware accelerator: GPU`.

It only reports what the session gives you. It downloads no dataset and trains no model.

## 1. Which GPU did this session get?

In [ ]:
!nvidia-smi

## 2. PyTorch and CUDA versions in this session

Record this output in `reports/experiment_log.md`. Colab upgrades its preinstalled
PyTorch over time, so the version that produced a result must be written down.

In [ ]:
import platform
import sys

import torch

print("python        :", platform.python_version())
print("torch         :", torch.__version__)
print("torch cuda    :", torch.version.cuda)
print("cudnn         :", torch.backends.cudnn.version())
print("cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit("No GPU in this session. Set Runtime > Change runtime type > GPU, then rerun.")

props = torch.cuda.get_device_properties(0)
print()
print("device        :", props.name)
print("total VRAM    : %.2f GB" % (props.total_memory / 1024**3))
print("compute cap   : %d.%d" % (props.major, props.minor))
print("bf16 support  :", torch.cuda.is_bf16_supported())

## 3. Does the GPU actually compute?

A correctness check plus a rough throughput number. The throughput tells you which
GPU class you were assigned (T4 / L4 / A100 differ a lot) and belongs in the
experiment log, because training time is one of the reported metrics.

In [ ]:
import time

import torch

device = torch.device("cuda")

# Correctness: a known-answer op, so a silently broken build is caught.
a = torch.ones(2, 3, device=device)
b = torch.full((2, 3), 2.0, device=device)
assert float((a + b).sum()) == 18.0
print("correctness   : OK")

# Throughput: fp32 matmul, averaged over a few runs after a warm-up.
n = 4096
x = torch.randn(n, n, device=device)
y = torch.randn(n, n, device=device)

for _ in range(3):
    x @ y
torch.cuda.synchronize()

runs = 10
start = time.perf_counter()
for _ in range(runs):
    x @ y
torch.cuda.synchronize()
elapsed = (time.perf_counter() - start) / runs

tflops = 2 * n**3 / elapsed / 1e12
print("matmul %dx%d : %.1f ms  (~%.1f TFLOP/s fp32)" % (n, n, elapsed * 1000, tflops))

## 4. How much VRAM can this session really use?

Allocates in steps until it fails, then releases everything. The last size that
succeeded is the practical memory budget, which decides the batch size and whether
VLM fine-tuning fits at all. Nothing is left allocated after this cell.

In [ ]:
import torch

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

blocks = []
block_gb = 0.5
allocated_gb = 0.0

try:
    while allocated_gb < 80:  # hard stop, larger than any Colab GPU
        n_floats = int(block_gb * 1024**3 / 4)
        blocks.append(torch.empty(n_floats, dtype=torch.float32, device="cuda"))
        allocated_gb += block_gb
except torch.cuda.OutOfMemoryError:
    pass

print("usable VRAM   : ~%.1f GB allocated before OOM" % allocated_gb)
print("peak reserved : %.2f GB" % (torch.cuda.max_memory_reserved() / 1024**3))

del blocks
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print("after cleanup : %.2f GB allocated" % (torch.cuda.memory_allocated() / 1024**3))

## 5. Session limits to be aware of

Disk space and RAM, plus a reminder that the session can disconnect at any time.

In [ ]:
!df -h / /content | sed -n '1,5p'
!free -g | sed -n '1,2p'

## 6. Record the result

Copy these four values into `reports/experiment_log.md` for this session:

| Field | Value |
| --- | --- |
| Date (UTC) | |
| GPU name and total VRAM | |
| torch version / CUDA version | |
| Usable VRAM before OOM | |

**Not yet:** no dataset download, no Drive mount, no training. Those come in later steps.